<a href="https://colab.research.google.com/github/mbmanoj904-hue/Generative-ai/blob/main/Langsmith.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# LANGGRAPH + RAG + 3 LLM + LANGSMITH TRACING
# GOOGLE COLAB - ONE CELL
# ============================================================

!pip install -q -U \
    langgraph \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    faiss-cpu \
    sentence-transformers \
    google-genai \
    langsmith


# ============================================================
# 1. IMPORTS
# ============================================================

import os
from typing import TypedDict

from google import genai
from google.colab import userdata

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langgraph.graph import StateGraph, START, END


# ============================================================
# 2. API KEYS
# ============================================================

GEMINI_API_KEY = (
   userdata.get("Geminiai")
)

LANGSMITH_API_KEY = userdata.get("Langsmith")

if not GEMINI_API_KEY:
    raise ValueError(
        "Gemini API key not found.\n"
        "Add GEMINI_API_KEY or Geminiai in Colab Secrets."
    )

if not LANGSMITH_API_KEY:
    raise ValueError(
        "LangSmith API key not found.\n"
        "Add LANGSMITH_API_KEY in Colab Secrets."
    )


# ============================================================
# 3. LANGSMITH CONFIGURATION
# ============================================================

os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"

# Change this to your desired LangSmith project name
os.environ["LANGCHAIN_PROJECT"] = "Fault-Tolerant-RAG-LangGraph"

# Optional
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"


print("LangSmith tracing enabled")
print("Project:", os.environ["LANGCHAIN_PROJECT"])


# ============================================================
# 4. GEMINI CLIENT
# ============================================================

client = genai.Client(
    api_key=GEMINI_API_KEY
)

MODEL_NAME = "gemini-3.8-flash"

print("Gemini model:", MODEL_NAME)


# ============================================================
# 5. KNOWLEDGE BASE
# ============================================================

knowledge = [

    """
    Artificial Intelligence (AI) is the field of computer science
    that focuses on creating machines and software capable of
    performing tasks that normally require human intelligence.

    Examples include learning, reasoning, problem solving,
    understanding language, computer vision and decision making.
    """,

    """
    Machine Learning (ML) is a subset of Artificial Intelligence.
    Machine Learning allows computers to learn patterns from data
    and make predictions or decisions without being explicitly
    programmed for every individual task.
    """,

    """
    Deep Learning is a subset of Machine Learning that uses
    artificial neural networks with multiple layers.
    It is widely used for image recognition, speech recognition,
    natural language processing and generative AI.
    """,

    """
    Large Language Models (LLMs) are AI models trained on large
    amounts of text data. They can understand and generate
    human-like text.
    """,

    """
    Retrieval-Augmented Generation (RAG) combines information
    retrieval with a language model.

    Relevant information is retrieved from a knowledge base
    and then provided to the language model to generate
    a grounded answer.
    """,

    """
    LangGraph is a framework for building stateful applications
    and agent workflows using graphs.

    Nodes perform tasks and edges define the workflow.
    """,

    """
    FAISS is a library for efficient similarity search of vectors.
    It is commonly used in RAG systems for retrieving documents
    based on semantic similarity.
    """
]


# ============================================================
# 6. CREATE DOCUMENTS
# ============================================================

documents = [
    Document(page_content=text)
    for text in knowledge
]

print("Documents:", len(documents))


# ============================================================
# 7. TEXT SPLITTING
# ============================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))


# ============================================================
# 8. EMBEDDINGS
# ============================================================

print("Loading embedding model...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ============================================================
# 9. FAISS VECTOR DATABASE
# ============================================================

print("Creating FAISS vector database...")

vector_db = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS database created")


# ============================================================
# 10. LANGGRAPH STATE
# ============================================================

class GraphState(TypedDict):

    question: str
    context: str

    llm1_answer: str
    llm2_answer: str
    llm3_answer: str

    llm1_status: str
    llm2_status: str
    llm3_status: str

    final_answer: str


# ============================================================
# 11. RETRIEVAL NODE
# ============================================================

def retrieve_node(state: GraphState):

    question = state["question"]

    print("\n==============================")
    print("RAG RETRIEVAL")
    print("==============================")

    try:

        results = vector_db.similarity_search(
            question,
            k=3
        )

        context = "\n\n".join(
            doc.page_content
            for doc in results
        )

        print("Retrieved:", len(results), "documents")

        return {
            "context": context
        }

    except Exception as e:

        print("Retrieval failed:", e)

        return {
            "context": ""
        }


# ============================================================
# 12. GEMINI FUNCTION
# ============================================================

def call_llm(prompt):

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    if not response.text:
        raise ValueError(
            "Empty response received from Gemini"
        )

    return response.text.strip()


# ============================================================
# 13. LLM 1
# ============================================================

def llm1_node(state: GraphState):

    print("\n==============================")
    print("LLM 1 - ANSWER GENERATOR")
    print("==============================")

    question = state["question"]
    context = state["context"]

    prompt = f"""
You are LLM 1.

Generate the best answer to the question using
the retrieved context.

Question:
{question}

Context:
{context}

Rules:
- Give a clear answer.
- Use the context.
- Do not invent information.
"""

    try:

        answer = call_llm(prompt)

        print("LLM 1 SUCCESS")

        return {
            "llm1_answer": answer,
            "llm1_status": "SUCCESS"
        }

    except Exception as e:

        print("LLM 1 FAILED:", e)

        return {
            "llm1_answer": "",
            "llm1_status": "FAILED"
        }


# ============================================================
# 14. LLM 2
# ============================================================

def llm2_node(state: GraphState):

    print("\n==============================")
    print("LLM 2 - CHECKER")
    print("==============================")

    question = state["question"]
    context = state["context"]
    answer1 = state["llm1_answer"]

    if answer1:

        previous = answer1

    else:

        previous = "LLM 1 failed to produce an answer."

    prompt = f"""
You are LLM 2, the checker and corrector.

Question:
{question}

Context:
{context}

LLM 1 Answer:
{previous}

Check the answer for correctness.

If necessary:
- Correct mistakes.
- Add missing information.
- Remove incorrect information.

If LLM 1 failed, generate the answer yourself.

Return only the improved answer.
"""

    try:

        answer = call_llm(prompt)

        print("LLM 2 SUCCESS")

        return {
            "llm2_answer": answer,
            "llm2_status": "SUCCESS"
        }

    except Exception as e:

        print("LLM 2 FAILED:", e)

        return {
            "llm2_answer": "",
            "llm2_status": "FAILED"
        }


# ============================================================
# 15. LLM 3
# ============================================================

def llm3_node(state: GraphState):

    print("\n==============================")
    print("LLM 3 - FINAL ANSWER")
    print("==============================")

    question = state["question"]
    context = state["context"]

    answer1 = state["llm1_answer"]
    answer2 = state["llm2_answer"]

    if answer2:

        previous = answer2

    elif answer1:

        previous = answer1

    else:

        previous = "Both previous LLMs failed."

    prompt = f"""
You are LLM 3, the final answer generator.

Question:
{question}

Retrieved Context:
{context}

LLM 1 Answer:
{answer1 if answer1 else "FAILED"}

LLM 2 Answer:
{answer2 if answer2 else "FAILED"}

Best Previous Answer:
{previous}

Generate the final accurate answer.

Rules:
- Always try to answer.
- Use the retrieved context.
- Correct previous mistakes.
- Keep the answer simple and clear.
- Do not mention internal failures.
"""

    try:

        answer = call_llm(prompt)

        print("LLM 3 SUCCESS")

        return {
            "llm3_answer": answer,
            "llm3_status": "SUCCESS",
            "final_answer": answer
        }

    except Exception as e:

        print("LLM 3 FAILED:", e)

        # Local fallback
        if answer2:

            final = answer2

        elif answer1:

            final = answer1

        elif context:

            final = context

        else:

            final = "Unable to generate an answer."

        return {
            "llm3_answer": final,
            "llm3_status": "FAILED",
            "final_answer": final
        }


# ============================================================
# 16. CREATE LANGGRAPH
# ============================================================

workflow = StateGraph(GraphState)


workflow.add_node(
    "retrieve",
    retrieve_node
)

workflow.add_node(
    "llm1",
    llm1_node
)

workflow.add_node(
    "llm2",
    llm2_node
)

workflow.add_node(
    "llm3",
    llm3_node
)


# ============================================================
# 17. GRAPH EDGES
# ============================================================

workflow.add_edge(
    START,
    "retrieve"
)

workflow.add_edge(
    "retrieve",
    "llm1"
)

workflow.add_edge(
    "llm1",
    "llm2"
)

workflow.add_edge(
    "llm2",
    "llm3"
)

workflow.add_edge(
    "llm3",
    END
)


# ============================================================
# 18. COMPILE
# ============================================================

app = workflow.compile()

print("\nLangGraph compiled successfully")


# ============================================================
# 19. USER QUESTION
# ============================================================

question = input(
    "\nEnter your question: "
)


# ============================================================
# 20. RUN
# ============================================================

initial_state = {

    "question": question,
    "context": "",

    "llm1_answer": "",
    "llm2_answer": "",
    "llm3_answer": "",

    "llm1_status": "",
    "llm2_status": "",
    "llm3_status": "",

    "final_answer": ""
}


result = app.invoke(
    initial_state
)


# ============================================================
# 21. OUTPUT
# ============================================================

print("\n")
print("=" * 70)
print("LANGSMITH PROJECT")
print("=" * 70)

print(
    "Project:",
    os.environ["LANGCHAIN_PROJECT"]
)

print("\nLLM STATUS")

print(
    "LLM 1:",
    result["llm1_status"]
)

print(
    "LLM 2:",
    result["llm2_status"]
)

print(
    "LLM 3:",
    result["llm3_status"]
)


print("\n")
print("=" * 70)
print("FINAL ANSWER")
print("=" * 70)

print(
    result["final_answer"]
)

print("\n")
print("=" * 70)
print("TRACE YOUR RUNS IN LANGSMITH")
print("=" * 70)
print(
    "Open your LangSmith dashboard and select project:"
)

print(
    os.environ["LANGCHAIN_PROJECT"]
)

LangSmith tracing enabled
Project: Fault-Tolerant-RAG-LangGraph
Gemini model: gemini-3.8-flash
Documents: 7
Chunks: 7
Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded
Creating FAISS vector database...
FAISS database created

LangGraph compiled successfully

Enter your question: what is ai & ml

RAG RETRIEVAL
Retrieved: 3 documents

LLM 1 - ANSWER GENERATOR
LLM 1 FAILED: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

LLM 2 - CHECKER
LLM 2 SUCCESS

LLM 3 - FINAL ANSWER
LLM 3 FAILED: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


LANGSMITH PROJECT
Project: Fault-Tolerant-RAG-LangGraph

LLM STATUS
LLM 1: FAILED
LLM 2: SUCCESS
LLM 3: FAILED


FINAL ANSWER
**Artificial Intelligence (AI)** is a field of computer science focused on creating machines and software capable of performing tasks that typically require human intelligence, such as l